[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/096.%20The%20Resilient%20Network%20Design%20%28Robust%20Optimization%29/P96-Tier-1_executed.ipynb)

---


# 96. The Resilient Network Design (Robust Optimization)
## Tier 1: Mathematical Formulation and Mixed-Integer Programming

### Purpose and inputs

This notebook chooses which facilities to build before uncertainty is known, then determines shipment quantities separately for each disruption scenario. The objective is to minimize the largest total cost among the modeled scenarios, while meeting a minimum service-coverage requirement in every scenario.

The inputs are:

- Potential facilities, each with a location, capacity, and fixed construction cost.
- Demand zones, each with a location and baseline demand.
- Disruption scenarios, each describing facility failures, blocked routes, and demand multipliers.
- A service requirement and a penalty per unit of unmet demand.

The outputs are the selected facilities, scenario-specific shipments, unmet demand, service coverage, and costs. The optimization model is a mixed-integer linear program (MILP): its cost expressions and constraints are linear, while facility decisions are binary.

### Model assumptions and scope

- Facility construction decisions are made once and are shared by all scenarios. Shipment and unmet-demand decisions may adapt after a scenario is known.
- Each scenario is treated as a possible case against which the plan must be robust. Scenario probabilities are recorded for context but are not used in the worst-case objective.
- A failed facility has no usable capacity, and a blocked facility-to-zone route cannot carry flow.
- Demand must be either served or counted as unmet. At least 80% of total demand must be served in each scenario; a large penalty for unmet demand also remains in the objective.
- Transport cost is proportional to Euclidean distance and quantity shipped. Facility capacity and demand use the same quantity units. Coordinates are measured in distance units, and the transport-cost coefficient converts distance and shipped quantity into monetary cost.
- The model does not represent facility operating costs, route capacity limits, partial facility failures, travel-time uncertainty, or decisions made dynamically during a disruption.

### Solution sequence

1. Define facilities, demand zones, and uncertainty scenarios.
2. Calculate scenario-specific per-unit transport costs and mark unavailable routes.
3. Formulate a two-stage robust mixed-integer optimization model.
4. Solve the model with PuLP's CBC solver and inspect the selected facilities and scenario outcomes.
5. Plot the solution and evaluate how the worst-case cost changes across service requirements.

The mathematical model is the central method. Binary first-stage facility decisions are combined with continuous second-stage shipment and unmet-demand decisions. An auxiliary variable represents the maximum scenario cost, allowing the min-max objective to be written as a linear optimization model.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pulp import (
    COIN_CMD,
    LpMinimize,
    LpProblem,
    LpVariable,
    lpSum,
    value,
)

# These labels keep solver status codes readable without relying on status names
# being re-exported from a particular PuLP module.
LP_STATUS_NAMES = {
    0: "Not Solved",
    1: "Optimal",
    -1: "Infeasible",
    -2: "Unbounded",
    -3: "Undefined",
}
LP_STATUS_OPTIMAL = 1

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
class Facility:
    """A potential facility with a location, capacity, and construction cost."""

    def __init__(self, id, x, y, capacity, fixed_cost):
        self.id = id
        self.x = x
        self.y = y
        self.capacity = capacity
        self.fixed_cost = fixed_cost


class DemandZone:
    """A demand location with a baseline quantity."""

    def __init__(self, id, x, y, demand):
        self.id = id
        self.x = x
        self.y = y
        self.demand = demand


class Scenario:
    """A disruption case and its scenario-specific demand changes."""

    def __init__(
        self,
        id,
        probability,
        facility_failures,
        route_blocks,
        demand_multipliers,
    ):
        self.id = id
        self.probability = probability
        self.facility_failures = facility_failures
        self.route_blocks = route_blocks
        self.demand_multipliers = demand_multipliers

In [ ]:
facilities = [
    Facility(1, 10, 50, 800, 500000),
    Facility(2, 50, 40, 600, 400000),
    Facility(3, 20, 70, 900, 600000),
]

demand_zones = [
    DemandZone(1, 15, 45, 100),
    DemandZone(2, 35, 20, 150),
    DemandZone(3, 25, 40, 200),
]

scenarios = [
    Scenario(
        id=1,
        probability=0.6,
        facility_failures=[],
        route_blocks=[],
        demand_multipliers={1: 1.0, 2: 1.0, 3: 1.0},
    ),
    Scenario(
        id=2,
        probability=0.4,
        facility_failures=[2],
        route_blocks=[(1, 3)],
        demand_multipliers={1: 3.0, 2: 0.33, 3: 0.5},
    ),
]

SERVICE_REQUIREMENT = 0.80
UNMET_DEMAND_PENALTY = 1_000_000

print("Problem setup:")
print(f"- Potential facilities: {len(facilities)}")
print(f"- Demand zones: {len(demand_zones)}")
print(f"- Uncertainty scenarios: {len(scenarios)}")
print(
    "- Required service coverage in every scenario: "
    f"{SERVICE_REQUIREMENT:.0%}"
)

In [ ]:
def calculate_distance(facility, demand_zone):
    """Return Euclidean distance between a facility and a demand zone."""
    return math.sqrt(
        (facility.x - demand_zone.x) ** 2
        + (facility.y - demand_zone.y) ** 2
    )


def calculate_transport_cost(facility, demand_zone):
    """Return the monetary cost of shipping one unit along a route."""
    return calculate_distance(facility, demand_zone) * 0.1


# A scenario cost matrix stores per-unit costs; infinity marks an unusable route.
transport_costs = {}
for scenario in scenarios:
    costs = np.zeros((len(facilities), len(demand_zones)))

    for i, facility in enumerate(facilities):
        for j, demand in enumerate(demand_zones):
            facility_failed = facility.id in scenario.facility_failures
            route_blocked = (facility.id, demand.id) in scenario.route_blocks

            if facility_failed or route_blocked:
                costs[i, j] = math.inf
            else:
                costs[i, j] = calculate_transport_cost(facility, demand)

    transport_costs[scenario.id] = costs

print("Per-unit transport-cost matrices (infinite values indicate unavailable routes):")
for scenario_id, costs in transport_costs.items():
    print(f"\nScenario {scenario_id}:")
    print(costs)

## Mathematical model

### Decisions and scenario data

A facility decision has a long-term role: it determines which capacity is available regardless of which disruption occurs. Shipment and unmet-demand decisions have a different role: they describe how the chosen network responds after a particular scenario is known. Keeping these decisions separate represents the two stages of the planning problem.

Let $I$ be the set of potential facilities, $J$ the set of demand zones, and $S$ the set of disruption scenarios. The binary first-stage variable $y_i$ equals 1 if facility $i$ is built and 0 otherwise. For each scenario $s$, the second-stage variable $x_{ijs}\geq 0$ is the quantity shipped from facility $i$ to zone $j$, and $z_{js}\geq 0$ is the unmet demand in zone $j$. These second-stage decisions are also called recourse decisions because they respond to the realized scenario. The auxiliary variable $t$ bounds the total cost in every scenario.

The baseline demand at zone $j$ is $d_j$, and its multiplier in scenario $s$ is $m_{js}$, so scenario demand is $d_{js}=d_jm_{js}$. Facility $i$ has capacity $C_i$ and fixed construction cost $F_i$. For an available facility-to-zone route, the per-unit transport cost $c_{ijs}$ is 0.1 times the Euclidean distance. Let $A_s$ denote the set of available facility-to-zone routes in scenario $s$: a route is available only if its facility has not failed and the route is not blocked. Unavailable routes carry no flow. The unmet-demand penalty per unit is $M=1{,}000{,}000$.

### Robust objective and constraints

A worst-case objective evaluates a plan by its most expensive modeled scenario, rather than by an average. The auxiliary variable $t$ represents that maximum cost. Requiring $t$ to be at least each scenario's cost makes minimizing the maximum equivalent to minimizing a single variable subject to linear constraints.

$$
\begin{aligned}
\min\quad & t \\
\text{subject to}\quad
 t &\geq \sum_{i\in I}F_i y_i
 +\sum_{(i,j)\in A_s}c_{ijs}x_{ijs}
 +M\sum_{j\in J}z_{js},
 && s\in S, \\
\sum_{i\in I}x_{ijs}+z_{js} &= d_{js},
 && j\in J,\ s\in S, \\
\sum_{j\in J}x_{ijs} &\leq C_i y_i,
 && i\in I\text{ operating in }s, \\
x_{ijs} &= 0,
 && (i,j)\notin A_s,\ s\in S, \\
\sum_{i\in I}\sum_{j\in J}x_{ijs} &\geq \rho\sum_{j\in J}d_{js},
 && s\in S, \\
y_i &\in\{0,1\},\qquad x_{ijs}\geq 0,\qquad z_{js}\geq 0.
\end{aligned}
$$

Here, $\rho=0.80$ is the required service fraction. The first constraint bounds the cost in each scenario; minimizing $t$ therefore minimizes the largest of those costs. The demand-balance equation means every unit of scenario demand is either shipped or recorded as unmet. The capacity constraint limits total shipments from an operating facility to its capacity when built, and to zero when it is not built. Failed facilities are excluded from this operating-facility constraint and have all their shipment variables fixed to zero by the unavailable-route constraint.

The coverage constraint applies to total demand across all zones in a scenario; it requires at least the specified fraction to be shipped, but does not require the same service fraction in every individual zone. Because shipped demand plus unmet demand equals total demand, this coverage requirement is equivalent to limiting total unmet demand to at most $(1-\rho)\sum_{j\in J}d_{js}$. The penalty $M$ additionally makes unmet demand costly; it does not replace the explicit coverage constraint.

The fixed construction cost appears in every scenario's cost expression because the same facility decisions apply to all scenarios. Shipment and unmet-demand variables are scenario-specific because they represent recourse after the disruption is known. Scenario probabilities are not part of this min-max objective.

### Reading the formulation

A feasible solution is a set of values for all decision variables that satisfies every constraint. An optimal solution is a feasible solution with the smallest possible value of $t$. The solver must choose facility decisions and scenario-specific responses together: a facility may be worth building because it helps in a difficult scenario, even if it is not needed in another scenario.

The first constraint is repeated once per scenario, so the common construction plan must have a bounded cost in every modeled case. The demand-balance constraints are repeated for each zone and scenario, while capacity and route-availability constraints prevent the model from assigning shipments to unavailable capacity or routes. The service constraint aggregates shipments across all facilities and zones for each scenario.

### Connection to the implementation

The dictionary `y` stores the binary facility decisions; `x` and `z` store scenario-specific shipment and unmet-demand variables. Their indices use positions in the facility and demand-zone lists, while scenario data such as failures and blocked routes use the objects' IDs. The variable `max_cost` is $t$. The list `scenario_costs` contains each scenario's cost expression, and each expression is constrained to be no greater than `max_cost`. The `LpProblem` minimizes that bound. Infinite entries in `transport_costs` mark unavailable routes; their shipment variables are constrained to zero, and they are omitted from the transport-cost expression.

The model is a mixed-integer linear program: its objective and constraints are linear, while the facility decisions are binary. With $|I|$ facilities, $|J|$ demand zones, and $|S|$ scenarios, the formulation creates $|I|$ binary facility variables, $|I||J||S|$ shipment variables, and $|J||S|$ unmet-demand variables, plus the maximum-cost variable. Exact mixed-integer optimization can require exponential time in the worst case as the problem grows, although actual solve time depends on the formulation and solver. CBC searches for an optimal solution to this finite formulation.

In [ ]:
def solve_robust_network_design(
    facilities,
    demand_zones,
    scenarios,
    transport_costs,
    service_requirement,
    unmet_demand_penalty,
    announce=True,
):
    """Build and solve the two-stage robust facility-location model."""
    model = LpProblem("Resilient_Network_Design", LpMinimize)

    # These binary decisions are shared by all scenarios: construction occurs first.
    y = {
        i: LpVariable(f"y_{i}", lowBound=0, upBound=1, cat="Binary")
        for i in range(len(facilities))
    }

    # Shipment and unmet-demand decisions adapt to each scenario.
    x = {}
    z = {}
    for s_idx, scenario in enumerate(scenarios):
        for i in range(len(facilities)):
            for j in range(len(demand_zones)):
                x[(i, j, s_idx)] = LpVariable(
                    f"x_{i}_{j}_{s_idx}", lowBound=0
                )

        for j in range(len(demand_zones)):
            z[(j, s_idx)] = LpVariable(
                f"z_{j}_{s_idx}", lowBound=0
            )

    max_cost = LpVariable("max_cost", lowBound=0)
    scenario_costs = []

    for s_idx, scenario in enumerate(scenarios):
        fixed_cost = lpSum(
            facilities[i].fixed_cost * y[i]
            for i in range(len(facilities))
        )
        transport_cost = lpSum(
            transport_costs[scenario.id][i, j] * x[(i, j, s_idx)]
            for i in range(len(facilities))
            for j in range(len(demand_zones))
            if math.isfinite(transport_costs[scenario.id][i, j])
        )
        unmet_penalty = lpSum(
            unmet_demand_penalty * z[(j, s_idx)]
            for j in range(len(demand_zones))
        )

        total_cost = fixed_cost + transport_cost + unmet_penalty
        scenario_costs.append(total_cost)
        model += total_cost <= max_cost

    model += max_cost

    facility_index = {
        facility.id: i for i, facility in enumerate(facilities)
    }
    demand_index = {
        demand.id: j for j, demand in enumerate(demand_zones)
    }

    for s_idx, scenario in enumerate(scenarios):
        for j, demand in enumerate(demand_zones):
            actual_demand = (
                demand.demand
                * scenario.demand_multipliers.get(demand.id, 1.0)
            )
            model += (
                lpSum(x[(i, j, s_idx)] for i in range(len(facilities)))
                + z[(j, s_idx)]
                == actual_demand
            )

        for i, facility in enumerate(facilities):
            if facility.id in scenario.facility_failures:
                for j in range(len(demand_zones)):
                    model += x[(i, j, s_idx)] == 0
            else:
                model += (
                    lpSum(
                        x[(i, j, s_idx)]
                        for j in range(len(demand_zones))
                    )
                    <= facility.capacity * y[i]
                )

        for facility_id, demand_id in scenario.route_blocks:
            i = facility_index[facility_id]
            j = demand_index[demand_id]
            model += x[(i, j, s_idx)] == 0

        total_demand = sum(
            demand.demand
            * scenario.demand_multipliers.get(demand.id, 1.0)
            for demand in demand_zones
        )
        model += (
            lpSum(
                x[(i, j, s_idx)]
                for i in range(len(facilities))
                for j in range(len(demand_zones))
            )
            >= service_requirement * total_demand
        )

    if announce:
        print("Solving the robust network design model...")
    model.solve(COIN_CMD(msg=False))

    return model, y, x, z, max_cost


model, y_vars, x_vars, z_vars, max_cost_var = solve_robust_network_design(
    facilities,
    demand_zones,
    scenarios,
    transport_costs,
    service_requirement=SERVICE_REQUIREMENT,
    unmet_demand_penalty=UNMET_DEMAND_PENALTY,
)

In [ ]:
def analyze_results(
    model,
    y_vars,
    x_vars,
    z_vars,
    max_cost_var,
    facilities,
    demand_zones,
    scenarios,
    transport_costs,
    service_requirement,
    unmet_demand_penalty,
):
    """Report facility choices, scenario service, and scenario costs."""
    print("=" * 64)
    print("ROBUST NETWORK DESIGN RESULTS")
    print("=" * 64)
    status_name = LP_STATUS_NAMES.get(
        model.status, f"Unknown status ({model.status})"
    )
    print(f"\nSolution status: {status_name}")

    if model.status != LP_STATUS_OPTIMAL:
        raise RuntimeError("The optimization model did not solve to optimality.")

    selected_facilities = []
    total_fixed_cost = 0

    print("\nFacility decisions:")
    for i, facility in enumerate(facilities):
        if value(y_vars[i]) > 0.5:
            selected_facilities.append(i)
            total_fixed_cost += facility.fixed_cost
            decision = f"BUILT (fixed cost: ${facility.fixed_cost:,.0f})"
        else:
            decision = "NOT BUILT"
        print(f"  Facility {facility.id}: {decision}")

    print(f"\nTotal fixed cost: ${total_fixed_cost:,.0f}")
    print("\nScenario analysis:")

    scenario_records = []
    for s_idx, scenario in enumerate(scenarios):
        total_demand = 0.0
        served_demand = 0.0
        unmet_demand = 0.0

        print(f"\nScenario {scenario.id}:")
        print(f"  Probability (descriptive only): {scenario.probability}")
        print(f"  Failed facilities: {scenario.facility_failures}")
        print(f"  Blocked routes: {scenario.route_blocks}")

        for j, demand in enumerate(demand_zones):
            actual_demand = (
                demand.demand
                * scenario.demand_multipliers.get(demand.id, 1.0)
            )
            zone_served = sum(
                value(x_vars[(i, j, s_idx)]) or 0.0
                for i in range(len(facilities))
            )
            zone_unmet = value(z_vars[(j, s_idx)]) or 0.0

            total_demand += actual_demand
            served_demand += zone_served
            unmet_demand += zone_unmet
            print(
                f"  Demand zone {demand.id}: demand {actual_demand:.1f}, "
                f"served {zone_served:.1f}, unmet {zone_unmet:.1f}"
            )

        scenario_transport_cost = sum(
            transport_costs[scenario.id][i, j]
            * (value(x_vars[(i, j, s_idx)]) or 0.0)
            for i in range(len(facilities))
            for j in range(len(demand_zones))
            if math.isfinite(transport_costs[scenario.id][i, j])
        )
        service_coverage = (
            served_demand / total_demand if total_demand > 0 else 0.0
        )
        unmet_penalty = unmet_demand_penalty * unmet_demand
        scenario_total_cost = (
            total_fixed_cost + scenario_transport_cost + unmet_penalty
        )

        print(f"  Service coverage: {service_coverage:.1%}")
        print(f"  Transport cost: ${scenario_transport_cost:,.2f}")
        print(f"  Unmet-demand penalty: ${unmet_penalty:,.2f}")
        print(f"  Total scenario cost: ${scenario_total_cost:,.2f}")

        scenario_records.append(
            {
                "scenario_id": scenario.id,
                "service_coverage": service_coverage,
                "fixed_cost": total_fixed_cost,
                "transport_cost": scenario_transport_cost,
                "unmet_penalty": unmet_penalty,
                "total_cost": scenario_total_cost,
            }
        )

    worst_coverage_record = min(
        scenario_records, key=lambda record: record["service_coverage"]
    )
    maximum_scenario_cost = max(
        record["total_cost"] for record in scenario_records
    )

    assert worst_coverage_record["service_coverage"] >= (
        service_requirement - 1e-7
    )
    assert math.isclose(
        maximum_scenario_cost,
        value(max_cost_var),
        rel_tol=1e-6,
        abs_tol=0.01,
    )

    print("\nWorst-case summary:")
    print(
        f"  Minimum service coverage: "
        f"{worst_coverage_record['service_coverage']:.1%} "
        f"(Scenario {worst_coverage_record['scenario_id']})"
    )
    print(f"  Maximum scenario cost: ${value(max_cost_var):,.2f}")
    print(
        f"  Coverage requirement of {service_requirement:.0%} "
        "is satisfied in every scenario."
    )

    return (
        selected_facilities,
        worst_coverage_record["service_coverage"],
        value(max_cost_var),
        scenario_records,
    )


(
    selected_facilities,
    worst_coverage,
    max_cost,
    scenario_records,
) = analyze_results(
    model,
    y_vars,
    x_vars,
    z_vars,
    max_cost_var,
    facilities,
    demand_zones,
    scenarios,
    transport_costs,
    service_requirement=SERVICE_REQUIREMENT,
    unmet_demand_penalty=UNMET_DEMAND_PENALTY,
)

In [ ]:
def create_visualization(
    facilities,
    demand_zones,
    scenarios,
    selected_facilities,
    scenario_records,
    service_requirements,
    service_requirement,
    unmet_demand_penalty,
    worst_coverage,
    max_cost,
):
    """Plot the network, scenario outcomes, costs, and sensitivity cases."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "Robust Network Design Results",
        fontsize=16,
        fontweight="bold",
    )

    # Distinct marker labels make the facility and demand symbols interpretable.
    ax1 = axes[0, 0]
    ax1.set_title("Network layout and facility decisions", fontweight="bold")
    selected_plotted = False
    unselected_plotted = False
    demand_plotted = False

    for i, facility in enumerate(facilities):
        if i in selected_facilities:
            label = "Selected facility" if not selected_plotted else None
            ax1.scatter(
                facility.x,
                facility.y,
                color="green",
                marker="o",
                s=220,
                edgecolors="black",
                label=label,
                zorder=3,
            )
            selected_plotted = True
        else:
            label = "Not selected" if not unselected_plotted else None
            ax1.scatter(
                facility.x,
                facility.y,
                color="red",
                marker="x",
                s=180,
                label=label,
                zorder=3,
            )
            unselected_plotted = True

        ax1.annotate(
            f"F{facility.id}",
            (facility.x, facility.y),
            xytext=(5, 5),
            textcoords="offset points",
            fontweight="bold",
        )

    for demand in demand_zones:
        label = "Demand zone" if not demand_plotted else None
        ax1.scatter(
            demand.x,
            demand.y,
            color="blue",
            marker="s",
            s=150,
            edgecolors="black",
            label=label,
            zorder=2,
        )
        demand_plotted = True
        ax1.annotate(
            f"D{demand.id}",
            (demand.x, demand.y),
            xytext=(5, 5),
            textcoords="offset points",
        )

    ax1.set_xlabel("X coordinate (distance units)")
    ax1.set_ylabel("Y coordinate (distance units)")
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc="best")

    ax2 = axes[0, 1]
    ax2.set_title("Service coverage and transport cost", fontweight="bold")
    scenario_labels = [
        f"Scenario {record['scenario_id']}" for record in scenario_records
    ]
    positions = np.arange(len(scenario_records))
    width = 0.35
    coverages = [
        100 * record["service_coverage"] for record in scenario_records
    ]
    scenario_transport_costs = [
        record["transport_cost"] for record in scenario_records
    ]

    coverage_bars = ax2.bar(
        positions - width / 2,
        coverages,
        width,
        label="Service coverage (%)",
        color="steelblue",
    )
    ax2.set_xlabel("Disruption scenario")
    ax2.set_ylabel("Service coverage (%)", color="steelblue")
    ax2.tick_params(axis="y", labelcolor="steelblue")
    ax2.set_xticks(positions)
    ax2.set_xticklabels(scenario_labels)
    ax2.axhline(
        100 * service_requirement,
        color="red",
        linestyle="--",
        label=f"Required coverage ({service_requirement:.0%})",
    )

    ax2_cost = ax2.twinx()
    cost_bars = ax2_cost.bar(
        positions + width / 2,
        scenario_transport_costs,
        width,
        label="Transport cost ($)",
        color="darkorange",
        alpha=0.8,
    )
    ax2_cost.set_ylabel("Transport cost ($)", color="darkorange")
    ax2_cost.tick_params(axis="y", labelcolor="darkorange")
    ax2.grid(True, axis="y", alpha=0.3)
    coverage_line = ax2.lines[-1]
    ax2.legend(
        [coverage_bars, cost_bars, coverage_line],
        [
            "Service coverage (%)",
            "Transport cost ($)",
            f"Required coverage ({service_requirement:.0%})",
        ],
        loc="upper left",
        fontsize=8,
    )

    ax3 = axes[1, 0]
    ax3.set_title("Scenario cost breakdown", fontweight="bold")
    fixed_costs = [record["fixed_cost"] for record in scenario_records]
    transport_costs_by_scenario = [
        record["transport_cost"] for record in scenario_records
    ]
    unmet_penalties = [
        record["unmet_penalty"] for record in scenario_records
    ]
    breakdown_width = 0.25
    ax3.bar(
        positions - breakdown_width,
        fixed_costs,
        breakdown_width,
        label="Fixed costs",
    )
    ax3.bar(
        positions,
        transport_costs_by_scenario,
        breakdown_width,
        label="Transport costs",
    )
    ax3.bar(
        positions + breakdown_width,
        unmet_penalties,
        breakdown_width,
        label="Unmet-demand penalties",
    )
    ax3.set_xlabel("Disruption scenario")
    ax3.set_ylabel("Cost ($)")
    ax3.set_xticks(positions)
    ax3.set_xticklabels(scenario_labels)
    ax3.legend(loc="best")
    ax3.grid(True, axis="y", alpha=0.3)

    ax4 = axes[1, 1]
    ax4.set_title("Solved service-requirement sensitivity", fontweight="bold")
    sensitivity_costs = []
    sensitivity_statuses = []

    for requirement_percent in service_requirements:
        sensitivity_model, _, _, _, sensitivity_max_cost = (
            solve_robust_network_design(
                facilities,
                demand_zones,
                scenarios,
                transport_costs,
                service_requirement=requirement_percent / 100,
                unmet_demand_penalty=unmet_demand_penalty,
                announce=False,
            )
        )
        status = LP_STATUS_NAMES.get(
            sensitivity_model.status,
            f"Unknown status ({sensitivity_model.status})",
        )
        sensitivity_statuses.append(status)
        if sensitivity_model.status == LP_STATUS_OPTIMAL:
            sensitivity_costs.append(value(sensitivity_max_cost))
        else:
            sensitivity_costs.append(np.nan)

    ax4.plot(
        service_requirements,
        sensitivity_costs,
        marker="o",
        linewidth=2,
        label="Optimal worst-case cost",
    )
    ax4.axvline(
        100 * service_requirement,
        color="green",
        linestyle="--",
        label=f"Base requirement ({service_requirement:.0%})",
    )
    ax4.axhline(
        max_cost,
        color="red",
        linestyle=":",
        label="Base-case worst-case cost",
    )
    ax4.set_xlabel("Required service coverage (%)")
    ax4.set_ylabel("Optimal worst-case cost ($)")
    ax4.grid(True, alpha=0.3)
    ax4.legend(loc="best", fontsize=8)

    for requirement_percent, cost, status in zip(
        service_requirements, sensitivity_costs, sensitivity_statuses
    ):
        if not math.isfinite(cost):
            ax4.annotate(
                f"{status}",
                (requirement_percent, 0),
                xytext=(0, 5),
                textcoords="offset points",
                rotation=45,
                fontsize=7,
            )

    fig.tight_layout()
    plt.show()
    return fig


fig = create_visualization(
    facilities,
    demand_zones,
    scenarios,
    selected_facilities,
    scenario_records,
    service_requirements=[70, 75, 80, 85, 90, 95],
    service_requirement=SERVICE_REQUIREMENT,
    unmet_demand_penalty=UNMET_DEMAND_PENALTY,
    worst_coverage=worst_coverage,
    max_cost=max_cost,
)

## Interpreting the solution

The facility variables identify strategic investments that are common to every scenario. The shipment variables show how the network uses those facilities under each disruption, while unmet demand records any quantity that is not served. Scenario cost combines the shared fixed investment, per-unit transport costs multiplied by shipped quantities, and the unmet-demand penalty.

The reported maximum total cost is the objective value of the robust model: it is the largest modeled scenario cost for the selected plan. Minimum service coverage is reported separately because the model imposes its service requirement in every scenario. A facility can be selected even if it is not needed in the least disruptive scenario, since its value may come from enabling service under a disruption. If multiple facility and shipment plans achieve the same optimal worst-case cost, the solver may return any one of them; the reported plan is an optimal solution, not necessarily the unique one.

The sensitivity panel re-solves the same mathematical model at each listed service requirement. Each plotted cost therefore comes from an optimization solve, rather than from an assumed cost increase. A non-optimal solve is shown without a cost value; such a point should be interpreted using its solver status, not as a numerical cost estimate. In general, imposing a stricter feasible service requirement cannot reduce the optimal worst-case cost, because it narrows the feasible set.

### Strengths, limitations, and use

**Strengths**
- The model explicitly represents multiple disruption scenarios, facility failures, blocked routes, and scenario-dependent demand.
- The min-max objective protects against the most costly scenario in the specified set.
- The formulation exposes the trade-off between facility investment, transport cost, and unmet demand.
- The mixed-integer solver provides an optimal solution for the stated mathematical model when it reports optimal status.

**Limitations**
- Exact mixed-integer optimization can become computationally demanding as the number of facilities, zones, and scenarios increases.
- Robustness is only with respect to the scenarios included in the model; unmodeled disruptions are not covered by the guarantee.
- Results depend on the accuracy of capacities, costs, demand multipliers, and disruption assumptions.
- The model is a static planning representation and does not capture real-time network changes or operational dynamics.

This formulation is suitable for strategic planning on networks of a size that can be solved within available computational resources, and for studying how explicit disruption scenarios affect facility choices and service.